# Data Sources
1. pdf
2. csv
3. JSON
4. APIs
5. Excel
6. Databases

## 2. Extract and Validate
- Read content
- Checking formats
- handles errors

## 3. Transform and Enrich data
- Clean text
- normalize
- attach metadata

## 4. Store and Index
- Files
- Databases
- searches
- Vector stores

# 0. Setup

In [2]:
! pip install pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable


In [3]:
# 1. Big Picture

In [5]:
from pathlib import Path
import json
import re
import pandas as pd



In [6]:
documents = [
    {
        "doc_id": "leave-policy",
        "source": "employee_leave_policy.txt",
        "title": "Employee Leave Policy",
        "department": "People",
        "text": (
            "Employees may carry over up to five unused vacation days into the next calendar year. "
            "Carry-over days must be used by March 31. Sick leave is recorded separately and does not "
            "count toward the vacation carry-over limit."
        ),
    },
    {
        "doc_id": "expense-policy",
        "source": "business_expenses.txt",
        "title": "Business Expense Policy",
        "department": "Finance",
        "text": (
            "Employees must submit expense claims within 30 days of the purchase. "
            "Receipts are required for individual expenses above 25 dollars. "
            "A manager must approve every travel expense before reimbursement."
        ),
    },
    {
        "doc_id": "security-policy",
        "source": "account_security.txt",
        "title": "Account Security Policy",
        "department": "IT",
        "text": (
            "Multi-factor authentication is required for all administrator accounts. "
            "Passwords must never be shared. Report suspected phishing messages to the security team."
        ),
    },
]

In [7]:
df = pd.DataFrame(documents)

In [9]:
df[["doc_id", "title", "department", "text"]]

,doc_id,title,department,text
0,leave-policy,Employee Leave Policy,People,Employees may carry over up to five unused vac...
1,expense-policy,Business Expense Policy,Finance,Employees must submit expense claims within 30...
2,security-policy,Account Security Policy,IT,Multi-factor authentication is required for al...


# Ingest

In [14]:
def validate_documents(records):
    errors = []
    seen_ids = set()

    for i, record in enumerate(records):
        doc_id = record.get("doc_id")
        text = record.get("text")

    if not doc_id:
        errors.append(f"records {i}: missing doc_id")
    elif doc_id in seen_ids:
        errors.append(f"Records {i}: duplicate doc_id {doc_id!r}")
    else:
        seen_ids.add(doc_id)

    if not isinstance(text, str) or not text.strip():
        errors.append(f"Record {i}: text is missing or empty")

    return errors


In [15]:
errors = validate_documents(documents)

In [16]:
print("Validation errors: ", errors)

Validation errors:  []


In [17]:
assert errors == []

In [18]:
bad_records = [
    {"doc_id": "x", "text" : "A valid Record"},
    {"doc_id": "x", "text" : "A Duplicate id"},
    {"doc_id": "empty", "text" : "   "}
]

In [19]:
validate_documents(bad_records)

['Record 2: text is missing or empty']

In [20]:
# Function to load a text file
def load_txt(path, doc_id=None):
    path = Path(path)
    return {
        "doc_id" : doc_id or path.stem,
        "source" : path.name,
        "title" : path.stem.replace("_", " ").title(),
        "text" : path.read_text(encoding="utf-8"),
    }
    

In [21]:
text_record = [load_txt("quick_guide.txt")]

In [23]:
print("TXT:", len(text_record))

TXT: 1


In [ ]:
## Cleaning Text Data

# Assignment:
# Project - AI Customer Feedback Analyzer
- The application should accept customer feedback and return structured analysis that another application can consume.

## Requirements:
- Accept a customer review from the terminal
- Reject null / Empty input
  Classify the sentiment: Positive, Negative, or Neutral
- Identify the feedback category
- Provide a brief explanation and recommendation
- Return Structured data - JSON
- Report API failures without crashing unexpectedly
- Keep the API key outside of the source code